<a href="https://colab.research.google.com/github/00015775/learning-lab/blob/learn%2Fpytorch/pytorch/notebooks/01_experiment_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading `weights` and `layers` of pretrained models

In [1]:
import torchvision
import torch

In [2]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT

In [3]:
weights.transforms() # observing what transformations were applied for pretrained model
# it is important to apply the SAME transformations/preprocessing
# for our custom data that the pre-trained model will learn from
# for all splits train/eval/test

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [4]:
weights.value

Weights(url='https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth', transforms=functools.partial(<class 'torchvision.transforms._presets.ImageClassification'>, crop_size=224, resize_size=256, interpolation=<InterpolationMode.BICUBIC: 'bicubic'>), meta={'categories': ['tench', 'goldfish', 'great white shark', 'tiger shark', 'hammerhead', 'electric ray', 'stingray', 'cock', 'hen', 'ostrich', 'brambling', 'goldfinch', 'house finch', 'junco', 'indigo bunting', 'robin', 'bulbul', 'jay', 'magpie', 'chickadee', 'water ouzel', 'kite', 'bald eagle', 'vulture', 'great grey owl', 'European fire salamander', 'common newt', 'eft', 'spotted salamander', 'axolotl', 'bullfrog', 'tree frog', 'tailed frog', 'loggerhead', 'leatherback turtle', 'mud turtle', 'terrapin', 'box turtle', 'banded gecko', 'common iguana', 'American chameleon', 'whiptail', 'agama', 'frilled lizard', 'alligator lizard', 'Gila monster', 'green lizard', 'African chameleon', 'Komodo dragon', 'African crocodile'

In [5]:
device = torch.device("cuda" if torch.cuda.is_available()\
                      else "cpu")

weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT # DEFAULT means the best available weights
model_0 = torchvision.models.efficientnet_b0(weights=weights).to(device)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 230MB/s]


In [6]:
!pip install -q torchinfo

In [7]:
from torchinfo import summary

In [8]:
model_0

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [9]:
summary(model_0, (1,3,64,56))
# since it uses AdaptiveAvgPool, which can take inputs of any shape
# as long as channel dimensions are the same

Layer (type:depth-idx)                                  Output Shape              Param #
EfficientNet                                            [1, 1000]                 --
├─Sequential: 1-1                                       [1, 1280, 2, 2]           --
│    └─Conv2dNormActivation: 2-1                        [1, 32, 32, 28]           --
│    │    └─Conv2d: 3-1                                 [1, 32, 32, 28]           864
│    │    └─BatchNorm2d: 3-2                            [1, 32, 32, 28]           64
│    │    └─SiLU: 3-3                                   [1, 32, 32, 28]           --
│    └─Sequential: 2-2                                  [1, 16, 32, 28]           --
│    │    └─MBConv: 3-4                                 [1, 16, 32, 28]           1,448
│    └─Sequential: 2-3                                  [1, 24, 16, 14]           --
│    │    └─MBConv: 3-5                                 [1, 24, 16, 14]           6,004
│    │    └─MBConv: 3-6                              

In [10]:
torch.nn.AdaptiveAvgPool2d # allows to take input images of any size

torch.nn.modules.pooling.AdaptiveAvgPool2d

In [11]:
for param in model_0.features.parameters():
  print(param.shape)

torch.Size([32, 3, 3, 3])
torch.Size([32])
torch.Size([32])
torch.Size([32, 1, 3, 3])
torch.Size([32])
torch.Size([32])
torch.Size([8, 32, 1, 1])
torch.Size([8])
torch.Size([32, 8, 1, 1])
torch.Size([32])
torch.Size([16, 32, 1, 1])
torch.Size([16])
torch.Size([16])
torch.Size([96, 16, 1, 1])
torch.Size([96])
torch.Size([96])
torch.Size([96, 1, 3, 3])
torch.Size([96])
torch.Size([96])
torch.Size([4, 96, 1, 1])
torch.Size([4])
torch.Size([96, 4, 1, 1])
torch.Size([96])
torch.Size([24, 96, 1, 1])
torch.Size([24])
torch.Size([24])
torch.Size([144, 24, 1, 1])
torch.Size([144])
torch.Size([144])
torch.Size([144, 1, 3, 3])
torch.Size([144])
torch.Size([144])
torch.Size([6, 144, 1, 1])
torch.Size([6])
torch.Size([144, 6, 1, 1])
torch.Size([144])
torch.Size([24, 144, 1, 1])
torch.Size([24])
torch.Size([24])
torch.Size([144, 24, 1, 1])
torch.Size([144])
torch.Size([144])
torch.Size([144, 1, 5, 5])
torch.Size([144])
torch.Size([144])
torch.Size([6, 144, 1, 1])
torch.Size([6])
torch.Size([144, 6, 

In [12]:
dir(model_0)

['T_destination',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_compiled_call_impl',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_impl',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_is_full_backward_hook',
 '_load_from_state_dict',
 '_load_state_dict_post_hooks',
 '_load_state_dict_pre_hooks',
 '_maybe_warn_non_full_backward_hook',
 '_modules',
 '_named_members',
 '_non_per

In [13]:
model_0.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

In [14]:
model_0.features

Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU(inplace=True)
  )
  (1): Sequential(
    (0): MBConv(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): SiLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (activation): SiLU(inplace=True)
          (scale_activation): Sigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), 

In [15]:
for param in model_0.features.parameters():
  param.requires_grad = False # freezing feature layers, as not to udpate them
  # use their weights as they are, without changing them

In [16]:
model_0.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

In [17]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# suppose we have 10 outputs only, so need to change the pre-trained model
# output shape from 1000 to 10
model_0.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280,
                    out_features=10,
                    bias=True).to(device)
)

In [18]:
model_0.classifier # see, the classifier layer is now overwritten

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=10, bias=True)
)

In [19]:
model_0

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [20]:
summary(model_0,
        (1,3,224,224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

# Trainable: False means that requires_grad=False
# so their weights do not change and stay as they are

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [1, 3, 224, 224]     [1, 10]              --                   Partial
├─Sequential (features)                                      [1, 3, 224, 224]     [1, 1280, 7, 7]      --                   False
│    └─Conv2dNormActivation (0)                              [1, 3, 224, 224]     [1, 32, 112, 112]    --                   False
│    │    └─Conv2d (0)                                       [1, 3, 224, 224]     [1, 32, 112, 112]    (864)                False
│    │    └─BatchNorm2d (1)                                  [1, 32, 112, 112]    [1, 32, 112, 112]    (64)                 False
│    │    └─SiLU (2)                                         [1, 32, 112, 112]    [1, 32, 112, 112]    --                   --
│    └─Sequential (1)                                        [1, 32, 112, 112]    [1, 1

In [21]:
# Since the requires_grad were set to False for feature layers
# those layers are not trainable now, and their weights will not be updated.
# hence there are only 12,810 trainable parameters in the classifer layer

> **Note**: The more trainable parameters a model has, the more compute power/longer it takes to train. Freezing the base layers of our model and leaving it with less trainable parameters means our model should train quite quickly. This is one huge benefit of transfer learning, taking the already learned parameters of a model trained on a problem similar to yours and only tweaking the outputs slightly to suit your problem.

In [22]:
weights.transforms()

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [23]:
from torchvision import transforms
from typing import List, Tuple
import matplotlib.pyplot as plt
from PIL import Image

def pred_and_plot_image(model: torch.nn.Module,
                        image_path: str,
                        class_names: List[str],
                        image_size: Tuple[int, int] = (224, 224),
                        transform: torchvision.transforms = None,
                        device: torch.device = device):

  img = Image.open(image_path)

  if transform is not None:
    image_transform = transform
  else:
    image_transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

  model.to(device)

  model.eval()
  with torch.inference_mode():
    transformed_image = image_transform(img).unsqueeze(dim=0) # creates extra batch dimension

    transformed_image = transformed_image.to(device)

    # values are not within certain range, nor intuitively understandle to properly compare them
    raw_logits = model(transformed_image)

    # raw logits -> probabilities
    pred_prob = torch.softmax(raw_logits, dim=1) # MUST CHOOSE CLASS DIMENSION
    # AVOID CHOOSING BATCH DIMENSION FOR SOFTMAX AND ARGMAX
    # CHOOSE CLASS DIMENSION

    # chooses the class index with the highest probability
    pred_label = torch.argmax(pred_prob, dim=1)

    plt.figure()
    plt.imshow(img)
    plt.title(f"Pred: {class_names[pred_label]} | Prob: {pred_prob.max() * 100:.3f}%")
    plt.axis(False)
    plt.show()



Models trained from scratch on small datasets often achieve very high training accuracy but fail to generalize due to overfitting.
Pretrained models, on the other hand, leverage previously learned representations that improve generalization, typically resulting in better validation and real-world performance.
Therefore, when working with limited data, transfer learning is usually the preferred approach — provided the source domain is reasonably related.

## **Transfor Learning Hugging Face**: https://huggingface.co/blog/RDTvlokip/when-ai-learns-from-experience-like-you?

In [24]:
import torch

device = torch.device("cuda" if torch.cuda.is_available()\
                      else "cpu")

In [25]:
import torch
import torchvision
import torchvision.models as models
from torch import nn

class TransferLearningDemo:

  def feature_extraction(self, num_classes):
    """Strategy 1: Freeze all, train only classifer"""
    # the fastest training

    resnet50_01_weights = torchvision.models.ResNet50_Weights.DEFAULT

    model_0 = torchvision.models.resnet50(weights=resnet50_01_weights).to(device)

    # freeze all the layers, but except the classifer layer
    # since the original pretrained model was trained on different number of classes
    for param in model_0.parameters():
      param.requires_grad = False

    # overwrites the pre-trained self.fc layer with this fine-tuned one below
    model_0.fc = nn.Linear(model_0.fc.in_features, num_classes).to(device)

    return model_0


  def fine_tuning(self, num_classes):
    """Strategy 2: Unfreeze some layers"""
    # moderate training time

    resnet50_02_weights = torchvision.models.ResNet50_Weights.DEFAULT # DEFAULT means the best performing weights

    model_1 = torchvision.models.resnet50(weights=resnet50_02_weights).to(device)

    # Unfreezing only classifier and the last/deep layers
    # deep layers learn high-level features, that is why better to update their
    # weights to better align for our case
    for name, param in model_1.named_parameters():
      if "layer4" in name or "fc" in name:
        param.requires_grad = True
      else:
        param.requires_grad = False

    model_1.fc = nn.Linear(in_features=model_1.fc.in_features,
                           out_features=num_classes).to(device)

    return model_1

  def full_fine_tuning(self, num_classes):
    """Strategy 3: Unfreeze all layers."""
    # long training time, but still faster than training from scratch a new model

    resnet50_03_weights = torchvision.models.ResNet50_Weights.DEFAULT

    model_2 = torchvision.models.resnet50(weights=resnet50_03_weights)

    for param in model_2.parameters():
      param.requires_grad = True

    model_2.fc = nn.Linear(model_2.fc.in_features,
                           num_classes).to(device)

    return model_2


In [26]:
num_flower_classes = 5

strategy_01 = TransferLearningDemo().feature_extraction(num_flower_classes)

strategy_02 = TransferLearningDemo().fine_tuning(num_flower_classes)

strategy_03 = TransferLearningDemo().full_fine_tuning(num_flower_classes)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 128MB/s]


**The key concept:** Instead of learning from scratch, you **inherit knowledge** from a model trained on millions of images. The early layers (edges, textures, lines) are **universal** and work for almost any task. Only the final layers need **task-specific retraining**, because the learn/capture high-level features such as faces/objects.

# Transfer Learning strategies

<pre>
How much data do you have?
│
├─ <500 images
│  └─ Feature Extraction (freeze all)
│     Learning rate: 0.001
│     Epochs: 10-20
│
├─ 500-5k images
│  └─ Fine-tuning (freeze early layers)
│     Learning rate: 0.0001
│     Epochs: 20-50
│
└─ 5k+ images
   └─ Full Fine-tuning (unfreeze all)
      Learning rate: 0.00001
      Epochs: 50-100
</pre>


## Feature Extraction

<pre>
Strategy: Freeze ALL pre-trained layers, train only classifier

ImageNet Model (frozen)
    ↓
[Conv layers 1-5] → FROZEN
    ↓
Remove old classifier
    ↓
Add NEW classifier for your task
    ↓
Train ONLY new classifier

Use when: Very little data (<1k images)
Speed: Ultra fast (minutes to hours)
Performance: Good but limited
</pre>


## Fine-tuning

<pre>
Strategy: Unfreeze SOME layers, train with low learning rate

ImageNet Model
    ↓
[Conv layers 1-3] → FROZEN  (low-level features)
    ↓
[Conv layers 4-5] → TRAINABLE  (high-level features)
    ↓
[New classifier] → TRAINABLE
    ↓
Train with low LR (0.0001)

Use when: Medium data (1k-10k images)
Speed: Moderate (hours)
Performance: Excellent
</pre>


## Full Fine-tuning

<pre>
Strategy: Unfreeze ALL layers, train everything

ImageNet Model
    ↓
[ALL layers] → TRAINABLE
    ↓
Train with very low LR (0.00001)

Use when: Lots of data (10k+ images)
Speed: Slowest (days)
Performance: Maximum but risky
</pre>

### Inspecting `ResNet50` layers

In [27]:
resnet50_01_weights = torchvision.models.ResNet50_Weights.DEFAULT

model_0 = torchvision.models.resnet50(weights=resnet50_01_weights).to(device)

In [28]:
!pip install -q torchinfo

In [29]:
from torchinfo import summary

summary(model_0, (1,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Conv2d: 1-1                            [1, 64, 112, 112]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         128
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 256, 56, 56]          --
│    └─Bottleneck: 2-1                   [1, 256, 56, 56]          --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           4,096
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-6                    [1, 64, 56, 56]           --
│ 

In [30]:
model_0

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [31]:
for name, val in model_0.named_parameters():
  print(name, val.shape)

conv1.weight torch.Size([64, 3, 7, 7])
bn1.weight torch.Size([64])
bn1.bias torch.Size([64])
layer1.0.conv1.weight torch.Size([64, 64, 1, 1])
layer1.0.bn1.weight torch.Size([64])
layer1.0.bn1.bias torch.Size([64])
layer1.0.conv2.weight torch.Size([64, 64, 3, 3])
layer1.0.bn2.weight torch.Size([64])
layer1.0.bn2.bias torch.Size([64])
layer1.0.conv3.weight torch.Size([256, 64, 1, 1])
layer1.0.bn3.weight torch.Size([256])
layer1.0.bn3.bias torch.Size([256])
layer1.0.downsample.0.weight torch.Size([256, 64, 1, 1])
layer1.0.downsample.1.weight torch.Size([256])
layer1.0.downsample.1.bias torch.Size([256])
layer1.1.conv1.weight torch.Size([64, 256, 1, 1])
layer1.1.bn1.weight torch.Size([64])
layer1.1.bn1.bias torch.Size([64])
layer1.1.conv2.weight torch.Size([64, 64, 3, 3])
layer1.1.bn2.weight torch.Size([64])
layer1.1.bn2.bias torch.Size([64])
layer1.1.conv3.weight torch.Size([256, 64, 1, 1])
layer1.1.bn3.weight torch.Size([256])
layer1.1.bn3.bias torch.Size([256])
layer1.2.conv1.weight tor

## **How to implement transfer learning in PyTorch**: https://www.geeksforgeeks.org/deep-learning/how-to-implement-transfer-learning-in-pytorch/

In [32]:
import torch
from torchinfo import summary
from torch import nn, optim

import matplotlib.pyplot as plt

from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Dataset


resnet50_weights = models.ResNet50_Weights.DEFAULT # DEFAULT means the best performing weights/biases

resnet_model_1 = models.resnet50(weights=resnet50_weights)


In [33]:
class ModifiedResNet(nn.Module):
  def __init__(self):
    super(ModifiedResNet, self).__init__()
    self.resnet = torch.hub.load("pytorch/vision", "resnet50", pretrained=True)
    num_classes = 10
    self.resnet.fc = nn.Linear(resnet_model_1.fc.in_features, num_classes)

  def forward(self, x):
    return self.resnet

modified_resnet_model = ModifiedResNet()

Downloading: "https://github.com/pytorch/vision/zipball/main" to /root/.cache/torch/hub/main.zip


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 152MB/s]


In [34]:
for param in modified_resnet_model.parameters():
  param.requires_grad = False # freeze all layers

In [35]:
from torchvision.transforms.functional import pad

transform_train = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# preprocessing must be applied for all splits, in order to yield consistent accuracy
# must use the same preprocessing on data as pre-trained model was trained on
# augmentation is not applied for the testing set
transform_test = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [36]:
train_dataset = torchvision.datasets.MNIST(root="./data",
                                           train=True,
                                           download=True,
                                           transform=transform_train)

test_dataset = torchvision.datasets.MNIST(root="./data",
                                          train=False,
                                          download=False,
                                          transform=transform_test)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.05MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.76MB/s]


In [37]:
train_loader = DataLoader(train_dataset,
                          batch_size=128,
                          shuffle=True)

test_loader = DataLoader(test_dataset,
                         batch_size=128,
                         shuffle=False)

In [38]:
len(train_dataset), len(test_dataset)

(60000, 10000)

In [39]:
len(train_loader), len(test_loader)

(469, 79)

In [40]:
60000/128, 10000/128 # rounded up to (469, 79)

(468.75, 78.125)

In [41]:
type(modified_resnet_model)

__main__.ModifiedResNet

In [42]:
class CustomResNet(nn.Module):
  def __init__(self, pretrained_model: torch.nn.Module):
    super(CustomResNet, self).__init__()
    self.resnet = torch.hub.load("pytorch/vision", "resnet50", pretrained=True)
    self.features = nn.Sequential(*list(pretrained_model.children())[:-1])
    num_classes = 10
    self.classifer = nn.Linear(pretrained_model.fc.in_features, num_classes)

  def forward(self, x):
    x = self.features(x)
    x = x.view(x.size(0), -1)
    #print(x.shape) # for debugging if needed
    x = self.classifer(x)
    return x

custom_model = CustomResNet(resnet_model_1)

Using cache found in /root/.cache/torch/hub/pytorch_vision_main


In [43]:
from torchinfo import summary

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(custom_model.parameters(),
                             lr=0.001)

summary(custom_model, (1,3,32,32))

Layer (type:depth-idx)                        Output Shape              Param #
CustomResNet                                  [1, 10]                   25,557,032
├─Sequential: 1-1                             [1, 2048, 1, 1]           --
│    └─Conv2d: 2-1                            [1, 64, 16, 16]           9,408
│    └─BatchNorm2d: 2-2                       [1, 64, 16, 16]           128
│    └─ReLU: 2-3                              [1, 64, 16, 16]           --
│    └─MaxPool2d: 2-4                         [1, 64, 8, 8]             --
│    └─Sequential: 2-5                        [1, 256, 8, 8]            --
│    │    └─Bottleneck: 3-1                   [1, 256, 8, 8]            75,008
│    │    └─Bottleneck: 3-2                   [1, 256, 8, 8]            70,400
│    │    └─Bottleneck: 3-3                   [1, 256, 8, 8]            70,400
│    └─Sequential: 2-6                        [1, 512, 4, 4]            --
│    │    └─Bottleneck: 3-4                   [1, 512, 4, 4]           

In [44]:
for param in custom_model.resnet.layer4.parameters():
  param.requires_grad = True # unfreezing the final hidden layers


custom_model.train()

CustomResNet(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
      

In [45]:
from tqdm.auto import tqdm
num_epochs = 2
train_losses = []
train_correct = 0
train_total = 0

custom_model.to(device)

for epoch in tqdm(range(num_epochs)):
  custom_model.train()
  running_loss = 0.0
  for batch, (inputs, labels) in enumerate(train_loader, start=1):
    inputs, labels = inputs.to(device), labels.to(device)

    optimizer.zero_grad()

    raw_logits = custom_model(inputs)
    loss = criterion(raw_logits, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

    _, predicted = torch.max(raw_logits, 1)
    train_total += labels.size(0) # batch dimension
    train_correct += (predicted == labels).sum().item()

    #if batch % 128 == 0:
  print(f"Epoch {epoch+1}/{num_epochs} | Loss: {running_loss/len(train_loader)}")

  train_losses.append(running_loss / len(train_loader.dataset))
  train_accuracy = train_correct / train_total



print(f"Finished fine-tuning with {train_accuracy*100}% accuracy")


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1/2 | Loss: 0.16704669126224067
Epoch 2/2 | Loss: 0.05441024176367342
Finished fine-tuning with 97.00666666666666% accuracy


In [46]:
len(train_loader), len(train_loader.dataset)

(469, 60000)

In [47]:
# len(train_loader) gives the total number of batches
# len(train_loader.dataset) gives the total number of individual data points
# which a lot more than len(train_loader)

In [48]:
from tqdm.auto import tqdm
test_losses = []
correct = 0
total = 0

custom_model.to(device)
custom_model.eval()
for epoch in tqdm(range(num_epochs)):
  with torch.inference_mode():
    running_loss = 0.0

    for images, labels in test_loader:
      images, labels = images.to(device), labels.to(device)

      outputs = custom_model(images)

      loss = criterion(outputs, labels)

      running_loss += loss.item()

      _, predicted = torch.max(outputs.data, 1)

      total += labels.size(0)
      correct += (predicted==labels).sum().item()

    test_loss = running_loss / len(test_loader.dataset)
    test_losses.append(test_loss)

  print(f"Epoch {epoch+1} | Test Loss: {test_loss:.4f}")

test_accuracy = correct / total

print(f"Accuracy of the model on the test set: {test_accuracy*100}% accuracy")


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1 | Test Loss: 0.0003
Epoch 2 | Test Loss: 0.0003
Accuracy of the model on the test set: 99.03% accuracy


In [49]:
nn.ReLU(inplace=True)

ReLU(inplace=True)

In [50]:
!pip install -q timm

In [51]:
import torch
import timm

all_models = timm.list_models("*resnet*") # wildcard

len(all_models), all_models[:5]

(134,
 ['cspresnet50',
  'cspresnet50d',
  'cspresnet50w',
  'eca_resnet33ts',
  'ecaresnet26t'])

In [52]:
all_models = timm.list_models()

len(all_models), all_models[:5]

(1284,
 ['aimv2_1b_patch14_224',
  'aimv2_1b_patch14_336',
  'aimv2_1b_patch14_448',
  'aimv2_3b_patch14_224',
  'aimv2_3b_patch14_336'])

In [53]:
dir(timm)

['__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 'create_model',
 'data',
 'get_pretrained_cfg',
 'get_pretrained_cfg_value',
 'is_exportable',
 'is_model',
 'is_model_pretrained',
 'is_scriptable',
 'layers',
 'list_models',
 'list_modules',
 'list_pretrained',
 'model_entrypoint',
 'models',
 'set_exportable',
 'set_scriptable',
 'utils',
 'version']

In [54]:
resnet_model = timm.create_model("resnet50",
                                 pretrained=True,
                                 num_classes=14,
                                 cache_dir="./resnet_model_folder/")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

In [55]:
!du -h ./resnet_model_folder/

4.0K	./resnet_model_folder/.locks/models--timm--resnet50.a1_in1k
8.0K	./resnet_model_folder/.locks
8.0K	./resnet_model_folder/models--timm--resnet50.a1_in1k/refs
8.0K	./resnet_model_folder/models--timm--resnet50.a1_in1k/snapshots/767268603ca0cb0bfe326fa87277f19c419566ef
12K	./resnet_model_folder/models--timm--resnet50.a1_in1k/snapshots
98M	./resnet_model_folder/models--timm--resnet50.a1_in1k/blobs
98M	./resnet_model_folder/models--timm--resnet50.a1_in1k
98M	./resnet_model_folder/


In [56]:
sum([p.numel() for p in resnet_model.parameters()])

23536718

In [57]:
summary(resnet_model, (1,3,45,45))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 14]                   --
├─Conv2d: 1-1                            [1, 64, 23, 23]           9,408
├─BatchNorm2d: 1-2                       [1, 64, 23, 23]           128
├─ReLU: 1-3                              [1, 64, 23, 23]           --
├─MaxPool2d: 1-4                         [1, 64, 12, 12]           --
├─Sequential: 1-5                        [1, 256, 12, 12]          --
│    └─Bottleneck: 2-1                   [1, 256, 12, 12]          --
│    │    └─Conv2d: 3-1                  [1, 64, 12, 12]           4,096
│    │    └─BatchNorm2d: 3-2             [1, 64, 12, 12]           128
│    │    └─ReLU: 3-3                    [1, 64, 12, 12]           --
│    │    └─Conv2d: 3-4                  [1, 64, 12, 12]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 12, 12]           128
│    │    └─Identity: 3-6                [1, 64, 12, 12]           --
│ 

In [58]:
resnet_model.fc

Linear(in_features=2048, out_features=14, bias=True)

In [59]:
resnet_model.fc.out_features

14

In [60]:
for name, param in resnet_model.named_parameters():
  print(name, param.shape)

conv1.weight torch.Size([64, 3, 7, 7])
bn1.weight torch.Size([64])
bn1.bias torch.Size([64])
layer1.0.conv1.weight torch.Size([64, 64, 1, 1])
layer1.0.bn1.weight torch.Size([64])
layer1.0.bn1.bias torch.Size([64])
layer1.0.conv2.weight torch.Size([64, 64, 3, 3])
layer1.0.bn2.weight torch.Size([64])
layer1.0.bn2.bias torch.Size([64])
layer1.0.conv3.weight torch.Size([256, 64, 1, 1])
layer1.0.bn3.weight torch.Size([256])
layer1.0.bn3.bias torch.Size([256])
layer1.0.downsample.0.weight torch.Size([256, 64, 1, 1])
layer1.0.downsample.1.weight torch.Size([256])
layer1.0.downsample.1.bias torch.Size([256])
layer1.1.conv1.weight torch.Size([64, 256, 1, 1])
layer1.1.bn1.weight torch.Size([64])
layer1.1.bn1.bias torch.Size([64])
layer1.1.conv2.weight torch.Size([64, 64, 3, 3])
layer1.1.bn2.weight torch.Size([64])
layer1.1.bn2.bias torch.Size([64])
layer1.1.conv3.weight torch.Size([256, 64, 1, 1])
layer1.1.bn3.weight torch.Size([256])
layer1.1.bn3.bias torch.Size([256])
layer1.2.conv1.weight tor

In [61]:
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 10)

In [62]:
resnet_model.fc

Linear(in_features=2048, out_features=10, bias=True)

In [63]:
resnet_model.num_classes

14

In [64]:
len(timm.list_models(pretrained=True))

1699

In [65]:
len(timm.list_models(pretrained=False))

1284

## **Transfer Learning with PyTorch**: https://arminnorouzi.github.io/posts/2023/05/blog-post-10/

In [66]:
torch.__version__

'2.9.0+cu126'

In [67]:
!nvidia-smi

Thu Feb  5 07:28:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P0             28W /   70W |    1174MiB /  15360MiB |      6%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [68]:
import torch
import torchvision

In [69]:
torch.__version__

'2.9.0+cu126'

In [70]:
torchvision.__version__

'0.24.0+cu126'

In [71]:
import matplotlib.pyplot as plt
from torch import nn
from torchvision import transforms
from torchvision import datasets, transforms

try:
  from torchinfo import summary
except:
  !pip install -q torchinfo
  from torchinfo import summary

In [72]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int=NUM_WORKERS
):

  train_data = datasets.ImageFolder(train_dir, transform=transforms)
  test_data = datasets.ImageFolder(test_dir, transform=transforms)

  class_names = train_data.classes # list of classes

  train_dataloader = DataLoader(
      train_data,
      batch_size=batch_size,
      shuffle=True,
      num_workers=num_workers,
      pin_memory=True
  )

  test_dataloader = DataLoader(
      test_data,
      batch_size=batch_size,
      shuffle=False,
      num_workers=num_workers,
      pin_memory=True
  )

  return train_dataloader, test_dataloader, class_names


In [73]:
device

device(type='cuda')

In [74]:
nn.BCEWithLogitsLoss # takes raw logits and applies softmax
nn.BCELoss # takes only softmax probabilities

torch.nn.modules.loss.BCELoss

In [75]:
import torch
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device = device) -> Tuple[float, float]:

  model.train()

  train_loss, train_acc = 0, 0

  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    y_pred = model(X)

    loss = loss_fn(y_pred, y)
    train_loss += loss.item() # cumulatively adding loss values from each FFN

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # raw logits -> probabilities
    y_pred_prob = torch.softmax(y_pred, dim=1) # along CLASS DIMENSION

    # probabilites -> target class indexes
    y_pred_class = torch.argmax(y_pred_prob, dim=1) # along CLASS DIMENSION

    # now dimensions both match after softmax and argmax
    train_acc += (torch.eq(y_pred_class, y).sum().item())/len(y_pred)

  train_loss = train_loss / len(dataloader)
  train_acc = train_acc / len(dataloader)

  return train_loss, train_acc


In [76]:
def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device = device) -> Tuple[float, float]:
  model.eval()

  test_loss, test_acc = 0, 0

  with torch.inference_mode():
    for batch, (X, y) in enumerate(dataloader):
      X, y = X.to(device), y.to(device)

      test_pred_logits = model(X)

      loss = loss_fn(test_pred_logits, y)
      test_loss += loss.item()

      test_pred_labels = test_pred_logits.argmax(dim=1)
      test_acc += (torch.eq(test_pred_labels, y).sum().item())/len(test_pred_labels)


  test_loss = test_loss / len(dataloader)
  test_acc = test_acc / len(dataloader)
  return test_loss, test_acc


In [77]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
weights

EfficientNet_B0_Weights.IMAGENET1K_V1

In [79]:
auto_transforms = weights.transforms()
auto_transforms

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [87]:
summary(model_0,
        (1,3,45,45),
        col_names=["input_size", "output_size", "num_params", "trainable"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Trainable
ResNet                                   [1, 3, 45, 45]            [1, 1000]                 --                        True
├─Conv2d: 1-1                            [1, 3, 45, 45]            [1, 64, 23, 23]           9,408                     True
├─BatchNorm2d: 1-2                       [1, 64, 23, 23]           [1, 64, 23, 23]           128                       True
├─ReLU: 1-3                              [1, 64, 23, 23]           [1, 64, 23, 23]           --                        --
├─MaxPool2d: 1-4                         [1, 64, 23, 23]           [1, 64, 12, 12]           --                        --
├─Sequential: 1-5                        [1, 64, 12, 12]           [1, 256, 12, 12]          --                        True
│    └─Bottleneck: 2-1                   [1, 64, 12, 12]           [1, 256, 12, 12]          --                        True
│    │ 